## Lab 2: Groq and Gemini model comparison

Groq's `openai/gpt-oss-120b` creates a challenging question, `openai/gpt-oss-20b` and Gemini answer it, and `openai/gpt-oss-120b` judges the responses.

In [10]:
import json
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [11]:
# Load GROQ_API_KEY and GOOGLE_API_KEY from your .env file
load_dotenv(override=True)

groq_api_key = os.getenv("GROQ_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY is not set. Add it to your .env file.")
if not google_api_key:
    raise ValueError("GOOGLE_API_KEY is not set. Add it to your .env file.")

print(f"Groq API key found (starts with {groq_api_key[:4]})")
print(f"Google API key found (starts with {google_api_key[:4]})")

Groq API key found (starts with gsk_)
Google API key found (starts with AIza)


## Create the model clients

Both APIs provide OpenAI-compatible endpoints. Gemini needs a Google API key; the GPT-OSS calls use Groq.

In [12]:
groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
gemini = OpenAI(
    api_key=google_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

## Generate a challenging question with Groq

`openai/gpt-oss-120b` generates one concise, thought-provoking question for the two models to answer.

In [13]:
question_request = (
    "Create one challenging, nuanced question that tests reasoning and judgment. "
    "It must not be a mathematical puzzle. Ask for a succinct answer. "
    "Return only the question, with no explanation or answer."
)

response = groq.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": question_request}],
)
question = response.choices[0].message.content.strip()
display(Markdown(f"### Question\n\n{question}"))

### Question

You are the leader of a small team tasked with launching a product that will significantly benefit the community, but the only available funding comes from a corporate sponsor whose values conflict with the product’s core mission. The sponsor insists on a public endorsement that would compromise the product’s integrity, yet without the funding the project cannot proceed. How do you proceed? (Answer in one sentence.)

## Ask the two models

The candidates are Groq's `openai/gpt-oss-20b` and Google's `gemini-3.1-flash-lite`.

In [14]:
candidates = [
    ("Groq: openai/gpt-oss-20b", groq, "openai/gpt-oss-20b"),
    ("Google: gemini-3.1-flash-lite", gemini, "gemini-3.1-flash-lite"),
]

answers = {}
for label, client, model_name in candidates:
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": question}],
    )
    answers[label] = response.choices[0].message.content.strip()
    display(Markdown(f"## {label}\n\n{answers[label]}"))

## Groq: openai/gpt-oss-20b

I would confront the sponsor, insisting that any partnership must respect the product’s mission—offering a neutral, non‑endorsing public acknowledgment instead of a value‑conflicting endorsement—while simultaneously mobilizing a community‑funding campaign and pursuing other grant opportunities, and if those efforts fail, I would defer the launch until I can secure funding that aligns ethically.

## Google: gemini-3.1-flash-lite

I would decline the sponsorship and instead pursue alternative funding avenues—such as community crowdfunding, public grants, or impact-focused partnerships—to ensure the project’s mission remains uncompromised, as sacrificing our core integrity would ultimately invalidate the very benefit we aim to provide.

## Ask Groq to judge the answers

The judge receives the question and both answers without being told which model produced them.

In [17]:
anonymous_answers = "\n\n".join(
    f"Answer {number}:\n{answer}"
    for number, answer in enumerate(answers.values(), start=1)
)
judge_prompt = f"""
You are a rigorous judge evaluating answers to a question.

Question:
{question}

Candidate answers:
{anonymous_answers}

Rank the answers using accuracy, reasoning, relevance, and clarity.
Return only valid JSON in this exact form:
{{"ranking": [1, 2], "rationale": "brief explanation"}}
Use the answer numbers in `ranking`, best first.
"""

response = groq.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": judge_prompt}],
)
judge_result = response.choices[0].message.content.strip()
print(judge_result)

{
  "ranking": [1, 2],
  "rationale": "Answer 1 offers a nuanced, step‑by‑step plan that directly addresses the sponsor conflict while also preparing alternative funding and a fallback, showing stronger reasoning and relevance; Answer 2 is clear and correct but less detailed, simply rejecting the sponsor without exploring negotiation options."
}


In [18]:
# Convert the judge's JSON response into a readable ranking
judge_result = judge_result.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
judgement = json.loads(judge_result)
labels = list(answers.keys())

ranking_lines = ["## Results", ""]
for place, answer_number in enumerate(judgement["ranking"], start=1):
    ranking_lines.append(f"{place}. {labels[answer_number - 1]}")
ranking_lines.extend(["", f"**Judge rationale:** {judgement['rationale']}"])
display(Markdown("\n".join(ranking_lines)))

## Results

1. Groq: openai/gpt-oss-20b
2. Google: gemini-3.1-flash-lite

**Judge rationale:** Answer 1 offers a nuanced, step‑by‑step plan that directly addresses the sponsor conflict while also preparing alternative funding and a fallback, showing stronger reasoning and relevance; Answer 2 is clear and correct but less detailed, simply rejecting the sponsor without exploring negotiation options.

## Exercise

Try a different question, change the evaluation criteria, or add another model and update the expected ranking format.